[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/padillacm/IA-practice/blob/master/recsys-course/19_elite_playbook/19_proyecto_debugging_produccion.ipynb)

# Proyecto 19 · *Pager duty* de recsys: depura CineMatch v2 en producción

**Nivel:** 🔴 Experto · **Duración:** 6–10 h · **GPU:** no necesaria (CPU de Colab; `FAST_DEV_RUN=False` tarda ~15–25 min en total) · **Unidades Colab:** < 1

**Prerrequisitos:** lección 19 (§3, §4, §6), módulos 16 (serving) y 17 (monitoreo).

## 📟 Contexto: son las 03:12 y suena el *pager*

> **[ALERTA P1] cinematch-home · conversiones por petición −10 % vs semana anterior (48 h sostenido)**

Hace dos días el equipo lanzó **CineMatch v2**: un retriever *two-tower* nuevo (embeddings v2, *warm-start* desde v1)
y un ranker GBDT con contadores en tiempo real. En la revisión de lanzamiento, el ranker v2 tenía **una AUC offline
excelente**. Online, las conversiones («el usuario terminó la película») han caído por debajo de las de v1.

Eres la persona de guardia. Tienes el código de producción, los logs de los últimos días y un **gemelo digital** del
tráfico (en la vida real no lo tendrías: aquí sirve para que puedas **medir el impacto real** de cada bug). El equipo
sospecha que hay **varios** problemas a la vez: se han identificado **6 bugs plantados**, todos realistas y todos
vistos en sistemas de verdad. Tu trabajo, como en un incidente real: **detectar** cada bug con una herramienta de
diagnóstico (no leyendo el código: en producción el código tiene 200 000 líneas), **cuantificar** su impacto y
**arreglarlo**, y escribir el *postmortem*.

## 📦 Dataset
**MovieLens-100K** (GroupLens). Sus factores latentes (SVD) definen los gustos «reales» de los usuarios del gemelo
digital; sobre ellos se simulan 14 días de logs del sistema v1 (con un 10 % de exploración) y los días de producción de
v2. Sin red, se usa un MovieLens sintético con el mismo esquema (el notebook nunca se bloquea).

## ✅ Entregables y rúbrica (100 puntos)

| Bloque | Puntos | Criterio |
|---|---|---|
| 6 bugs × (detección 5 + cuantificación 5 + arreglo 5) | 90 | **Detección**: la herramienta implementada muestra la anomalía con un número (no vale «leí el código»). **Cuantificación**: Δ conv/petición al arreglar solo ese bug sobre la config de producción. **Arreglo**: el cambio de config/código y la herramienta vuelve a verde |
| *Postmortem* | 10 | Línea temporal, causa raíz de cada bug, por qué no lo vio la revisión de lanzamiento, acciones preventivas con *owner* |

**Objetivos numéricos** (el *scorecard* final los comprueba):
- conv/petición con todo arreglado ≥ **1,10 ×** la de la config de producción **y** ≥ **0,98 ×** la de v1;
- % de recomendaciones ya vistas < **1 %**;
- la AUC offline del pipeline arreglado es **honesta**: su diferencia con una evaluación sin fugas < 0,01.

## 🗺️ Plan
1. Carga el gemelo digital, los logs y el código de producción (dado).
2. Reproduce el incidente (dado).
3. Construye tu caja de herramientas de diagnóstico (**TODOs**).
4. Bug por bug, en el orden del árbol de triaje de la lección (§6): detecta → cuantifica → arregla.
5. *Scorecard* y *postmortem*.
6. ⛔ Solución de referencia completa.

In [ ]:
!pip install -q numpy pandas matplotlib scikit-learn scipy

In [ ]:
import os, io, zipfile, urllib.request, time, json, math, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from IPython.display import display

warnings.filterwarnings("ignore")
seed = 42
random.seed(seed); np.random.seed(seed)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

# LightGBM con 1 hilo: faiss-cpu trae su propia libgomp y dos runtimes OpenMP en el mismo proceso pueden bloquearse
# o ir muy lentos con muchos hilos. Con nuestros tamaños, 1–2 hilos bastan (sube LGB_THREADS si entrenas a escala).
LGB_THREADS = int(os.environ.get("LGB_THREADS", "1"))
DATA_DIR, ART_DIR = "data", "artifacts"
os.makedirs(DATA_DIR, exist_ok=True); os.makedirs(ART_DIR, exist_ok=True)
ML100K_URL = "https://files.grouplens.org/datasets/movielens/ml-100k.zip"
GENRES = ["unknown", "Action", "Adventure", "Animation", "Children", "Comedy", "Crime",
          "Documentary", "Drama", "Fantasy", "Film-Noir", "Horror", "Musical", "Mystery",
          "Romance", "Sci-Fi", "Thriller", "War", "Western"]


def synthetic_movielens(n_users=943, n_items=1682, n_ratings=100_000, seed=42):
    """Fallback sin red: imita el esquema de ML-100K (gustos por género + popularidad Zipf)."""
    rng = np.random.default_rng(seed)
    item_genres = (rng.random((n_items, len(GENRES))) < 0.12).astype(int)
    item_genres[np.arange(n_items), rng.integers(1, len(GENRES), n_items)] = 1
    user_taste = rng.dirichlet(np.ones(len(GENRES)) * 0.3, n_users)
    pop = 1.0 / np.arange(1, n_items + 1) ** 0.9
    pop = pop[rng.permutation(n_items)]
    rows, t0 = [], 874_724_710  # sept-1997, como ML-100K
    per_user = rng.multinomial(n_ratings, rng.dirichlet(np.ones(n_users)))
    for u in range(n_users):
        k = max(per_user[u], 20)
        p = pop * (item_genres @ user_taste[u] + 0.02)
        p /= p.sum()
        items = rng.choice(n_items, size=min(k, n_items // 2), replace=False, p=p)
        aff = item_genres[items] @ user_taste[u]
        r = np.clip(np.round(2.5 + 6 * aff + rng.normal(0, 0.8, len(items))), 1, 5)
        ts = t0 + np.sort(rng.integers(0, 215 * 86400, len(items)))
        rows.append(pd.DataFrame({"user_id": u + 1, "item_id": items + 1, "rating": r.astype(int), "ts": ts}))
    ratings = pd.concat(rows, ignore_index=True)
    items_df = pd.DataFrame(item_genres, columns=GENRES)
    items_df.insert(0, "title", [f"Película sintética {i + 1}" for i in range(n_items)])
    items_df.insert(0, "item_id", np.arange(1, n_items + 1))
    return ratings, items_df, "Sintético (esquema ML-100K)"


def load_ml100k():
    try:
        path = os.path.join(DATA_DIR, "ml-100k.zip")
        if not os.path.exists(path):
            urllib.request.urlretrieve(ML100K_URL, path)
        with zipfile.ZipFile(path) as z:
            ratings = pd.read_csv(z.open("ml-100k/u.data"), sep="\t",
                                  names=["user_id", "item_id", "rating", "ts"])
            raw = pd.read_csv(z.open("ml-100k/u.item"), sep="|", encoding="latin-1", header=None)
        items_df = raw[[0, 1] + list(range(5, 24))].copy()
        items_df.columns = ["item_id", "title"] + GENRES
        return ratings, items_df, "MovieLens-100K"
    except Exception as e:  # sin red / URL caída → nunca bloqueamos el notebook
        print("⚠️ No se pudo descargar ML-100K, uso datos sintéticos:", repr(e)[:120])
        return synthetic_movielens()


ratings, items_df, DATASET = load_ml100k()
ratings = ratings.sort_values("ts").reset_index(drop=True)
print(DATASET, ratings.shape, "usuarios:", ratings.user_id.nunique(), "ítems:", ratings.item_id.nunique())

In [ ]:
# ⚙️ EL GEMELO DIGITAL — es la «realidad» del proyecto: NO lo modifiques.
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import svds
from scipy.linalg import expm
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

FAST_DEV_RUN = True                       # ⇦ False para la entrega (más días de logs y de producción)
N_LOG_DAYS, N_ONLINE_DAYS = (14, 4) if FAST_DEV_RUN else (21, 7)
REQ_FRAC = 0.3                            # fracción de usuarios que visita CineMatch cada día
rng = np.random.default_rng(seed)

N_U, N_I = int(ratings.user_id.max()) + 1, int(items_df.item_id.max()) + 1
X_ = csr_matrix((np.ones(len(ratings), np.float32), (ratings.user_id, ratings.item_id)), shape=(N_U, N_I))
u_, s_, vt_ = svds(X_, k=16, random_state=seed)
U_TRUE, V_TRUE = u_ * np.sqrt(s_), vt_.T * np.sqrt(s_)
A_TRUE = U_TRUE @ V_TRUE.T
A_TRUE = ((A_TRUE - A_TRUE.mean()) / A_TRUE.std()).astype(np.float32)          # afinidad real usuario×película
pop_hist = np.asarray(X_.sum(0)).ravel()
pop_z = (np.log1p(pop_hist) - np.log1p(pop_hist).mean()) / np.log1p(pop_hist).std()
quality = 0.4 * pop_z + 0.8 * np.minimum(pop_z, 0) + 0.8 * rng.normal(size=N_I)   # la cola larga tiene mucho «relleno»
release = -rng.integers(30, 1500, N_I).astype(float)                              # día de alta en el catálogo
new_items = rng.choice(np.arange(1, N_I), size=int(0.12 * N_I), replace=False)
release[new_items] = rng.uniform(-3, N_LOG_DAYS + N_ONLINE_DAYS, len(new_items))  # estrenos durante la simulación
release[0] = 1e9                                                                  # el ítem 0 no existe
pop_feat = pop_z.copy(); pop_feat[new_items] = 0.0       # la «popularidad histórica» de un estreno no se conoce: se imputa la media
POS_BIAS = 1.0 / (1 + np.arange(10)) ** 0.7
CONV_DELAY_MEAN = 3.0                                                             # días entre el clic y «terminó la película»
history = {u: list(g) for u, g in ratings.groupby("user_id").item_id}
for u in range(1, N_U):
    history.setdefault(u, [])


def sigmoid(x):
    return 1 / (1 + np.exp(-x))


def true_click_logit(u, items, day):
    age = day - release[items]
    trend = np.where(age >= 0, 1.6 * np.exp(-np.clip(age, 0, None) / 4.0), -np.inf)   # los estrenos tiran mucho unos días
    return -4.3 + 1.1 * A_TRUE[u, items] + 0.3 * pop_z[items] + trend


def true_conv_prob(u, items):
    return sigmoid(-2.0 + 0.5 * A_TRUE[u, items] + 1.3 * quality[items])


def available(day):
    return np.where(release <= day)[0]


def _learned(noise, seed_):
    r = np.random.default_rng(seed_)
    Q, _ = np.linalg.qr(r.normal(size=(16, 16)))
    E = (V_TRUE + noise * V_TRUE.std() * r.normal(size=V_TRUE.shape)) @ Q
    return (E / (np.linalg.norm(E, axis=1, keepdims=True) + 1e-8)).astype(np.float32)


def _rotation(angle, seed_):
    W = np.random.default_rng(seed_).normal(size=(16, 16)); W = (W - W.T) / 2
    return expm(angle * W / np.linalg.norm(W, 2))


EMB = {"v2": _learned(0.6, 2)}                                  # embeddings de ítem de la torre nueva (v2)
_v1 = (EMB["v2"] + 0.125 * np.random.default_rng(3).normal(size=EMB["v2"].shape)) @ _rotation(1.0, 4)
EMB["v1"] = (_v1 / np.linalg.norm(_v1, axis=1, keepdims=True)).astype(np.float32)   # v2 se entrenó con warm-start desde v1


def user_embedding(hist, version):
    v = EMB[version][hist[-50:]].mean(0) if len(hist) else np.zeros(16, np.float32)
    return v / (np.linalg.norm(v) + 1e-8)


print(f"{DATASET}: {N_U - 1} usuarios, {N_I - 1} películas, {len(new_items)} estrenos · {N_LOG_DAYS} días de logs + {N_ONLINE_DAYS} de producción")

## 1. Los logs del sistema anterior (v1)

Cada fila es una impresión: usuario, película, día, instante `t`, posición, clic, conversión, instante de la conversión
(`conv_t`, infinito si no convirtió) y si era el **hueco de exploración** (1 de cada 10, aleatorio sobre el catálogo).

In [ ]:
def v1_policy(u, day, hist, k=10, r=rng):
    """CineMatch v1 (el sistema anterior): top-300 popular + estrenos, y 1 hueco de exploración aleatoria."""
    av = available(day); av = av[~np.isin(av, hist)]
    pool = av[(np.argsort(np.argsort(-pop_hist[av])) < 300) | (day - release[av] < 10)]
    sc = 1.0 * pop_z[pool] + 3.0 * (EMB["v1"][pool] @ user_embedding(hist, "v1"))
    if r is not None:
        sc = sc + r.gumbel(size=len(pool)) * 0.5
    shown = list(pool[np.argsort(-sc)[:k - (r is not None)]])
    explore = np.zeros(k, int)
    if r is not None:
        rest = av[~np.isin(av, shown)]; pos = int(r.integers(k))
        shown.insert(pos, int(r.choice(rest))); explore[pos] = 1
    return np.array(shown), explore


def generate_logs():
    rows, hist = [], {u: list(h) for u, h in history.items()}
    for day in range(N_LOG_DAYS):
        for u in rng.choice(np.arange(1, N_U), size=int(REQ_FRAC * N_U), replace=False):
            t = day + rng.random()
            shown, explore = v1_policy(u, day, hist[u])
            clk = rng.random(len(shown)) < POS_BIAS * sigmoid(true_click_logit(u, shown, day))
            conv = clk & (rng.random(len(shown)) < true_conv_prob(u, shown))
            delay = rng.exponential(CONV_DELAY_MEAN, len(shown))
            for p in range(len(shown)):
                rows.append((u, int(shown[p]), day, t, p, int(clk[p]), int(conv[p]), t + delay[p] if conv[p] else np.inf, int(explore[p])))
            hist[u] += [int(i) for i in shown[clk]]
    cols = ["user_id", "item_id", "day", "t", "position", "click", "conv", "conv_t", "explore"]
    return pd.DataFrame(rows, columns=cols), hist


logs, hist_after_logs = generate_logs()
print(f"logs: {len(logs):,} impresiones · CTR {logs.click.mean():.3f} · conversiones/clic {logs.conv.sum() / logs.click.sum():.2f} · exploración {logs.explore.mean():.1%}")
display(logs.head())

## 2. El código de producción de CineMatch v2

Esto es lo que corre en producción: contadores por ítem, features, retriever, pipeline de entrenamiento del ranker y
servicio con caché de usuario. **Los bugs están aquí dentro.** Puedes leerlo, pero la rúbrica exige que cada bug lo
**demuestre una herramienta** con un número: así es como se trabaja cuando el código es enorme y el incidente es ahora.

In [ ]:
# ============================  src/cinematch_v2.py  (código de PRODUCCIÓN)  ============================
PROD_CONFIG = dict(
    counter_window="inclusive",            # cómo se cortan las ventanas de los contadores en el pipeline de train
    hash_buckets=97,                       # tamaño de la tabla de contadores por ítem (None = exacta)
    age_unit_serving="hours",              # unidad con la que el servicio calcula la edad del título
    label_policy="observed",               # cómo se etiquetan los clics sin conversión todavía
    train_on="impressions_no_explore",     # qué filas de los logs usa el entrenamiento
    cache_key="user_id",                   # clave de la caché de usuario del servicio
    cache_invalidate_on_event=False,       # ¿se invalida la caché cuando el usuario hace clic?
)
FEATS = ["retr_score", "item_ctr_1d", "item_cvr_7d", "item_imps_7d", "item_age_days", "item_pop_hist", "user_n_hist", "position"]
T_TRAIN = float(N_LOG_DAYS)                # el pipeline de entrenamiento corre al final del último día de logs
SERVING_VERSION = "v2"


class Counters:
    """Contadores diarios por ítem (impresiones, clics, conversiones llegadas) en una tabla, opcionalmente hasheada."""
    def __init__(self, logs, n_days, hash_buckets=None):
        self.m = hash_buckets
        nb = hash_buckets or N_I
        self.imps, self.clk, self.cnv = (np.zeros((n_days + 2, nb)) for _ in range(3))
        b = self.bucket(logs.item_id.values)
        np.add.at(self.imps, (logs.day.values, b), 1); np.add.at(self.clk, (logs.day.values, b), logs.click.values)
        cv = logs[np.isfinite(logs.conv_t) & (logs.conv_t < n_days)]
        np.add.at(self.cnv, (np.floor(cv.conv_t.values).astype(int), self.bucket(cv.item_id.values)), 1)

    def bucket(self, items):
        items = np.asarray(items, dtype=np.int64)
        return items if self.m is None else (items * 2654435761 % 2**32) % self.m

    def asof(self, items, day, inclusive):
        """CTR del último día, CVR y log-impresiones de los últimos 7 días."""
        b, s_ = self.bucket(items), int(inclusive)
        ctr = (self.clk[day - 1 + s_, b] + 0.5) / (self.imps[day - 1 + s_, b] + 10.0)
        lo, hi = max(0, day - 7 + s_), day + s_
        cvr = (self.cnv[lo:hi, b].sum(0) + 1.0) / (self.clk[lo:hi, b].sum(0) + 4.0)
        return ctr, cvr, np.log1p(self.imps[lo:hi, b].sum(0))


def features(items, day, hist, emb_user, cfg, counters, serving, position=None):
    ctr, cvr, imps = counters.asof(items, int(day), inclusive=(cfg["counter_window"] == "inclusive") and not serving)
    age = day - release[items]
    if serving and cfg["age_unit_serving"] == "hours":
        age = age * 24
    return pd.DataFrame({"retr_score": EMB["v2"][items] @ emb_user, "item_ctr_1d": ctr, "item_cvr_7d": cvr,
                         "item_imps_7d": imps, "item_age_days": age, "item_pop_hist": pop_feat[items],
                         "user_n_hist": np.full(len(items), np.log1p(len(hist))),
                         "position": position if position is not None else np.zeros(len(items))})


def retrieve(emb_user, day, seen, k=100):
    av = available(day); av = av[~np.isin(av, seen)]
    sc = EMB["v2"][av] @ emb_user
    top = np.argpartition(-sc, min(k, len(av) - 1))[:k]
    return av[top[np.argsort(-sc[top])]]


def make_labels(L, cfg, T=T_TRAIN):
    """Etiqueta = «terminó la película» (conversión) observada en el instante T del pipeline."""
    L = L.copy()
    L["label"] = ((L.conv == 1) & (L.conv_t <= T)).astype(float)
    if cfg["label_policy"] == "soft":
        L = soft_labels(L, T)              # ← corrección (la implementas tú en el bug 5)
    return L


def build_training_frame(L, cfg, counters, logs_all):
    frames = []
    clicks = {u: g for u, g in logs_all[logs_all.click == 1].groupby("user_id")}
    if cfg["train_on"] == "impressions_no_explore":
        L = L[L.explore == 0]                          # «el tráfico de exploración ensucia las métricas»
    for (u, day), g in L.groupby(["user_id", "day"]):
        c_u = clicks.get(u)
        h = history[u] + ([] if c_u is None else c_u.item_id[c_u.day < day].tolist())
        f = features(g.item_id.values, day, h, user_embedding(h, "v2"), cfg, counters, serving=False, position=g.position.values)
        f["label"] = g.label.values
        f["w"] = np.where(g.explore.values == 1, EXPLORE_WEIGHT, 1.0) if cfg["train_on"] == "all_ipw" else 1.0
        f["user_id"], f["day"], f["item_id"], f["t"] = u, day, g.item_id.values, g.t.values
        frames.append(f)
    return pd.concat(frames, ignore_index=True)


def expand_soft(frame):
    """Etiquetas en (0,1) → dos filas ponderadas (y=1 con peso p, y=0 con peso 1−p)."""
    soft = (frame.label > 0) & (frame.label < 1)
    hard = frame[~soft].assign(label=frame.label[~soft].astype(int))
    if not soft.any():
        return hard
    pos = frame[soft].assign(w=lambda d: d.w * d.label, label=1)
    neg = frame[soft].assign(w=lambda d: d.w * (1 - d.label), label=0)
    return pd.concat([hard, pos, neg], ignore_index=True)


def train_ranker(frame):
    fr = expand_soft(frame)
    m = HistGradientBoostingClassifier(max_iter=200, learning_rate=0.06, max_leaf_nodes=15, min_samples_leaf=40, random_state=seed)
    return m.fit(fr[FEATS], fr.label, sample_weight=fr.w)


def run_pipeline(cfg):
    """Pipeline nocturno: contadores → etiquetas → frame de train → ranker → evaluación offline en el último día."""
    counters = Counters(logs, N_LOG_DAYS + N_ONLINE_DAYS, cfg["hash_buckets"])
    L = make_labels(logs, cfg)
    val_day = N_LOG_DAYS - 1
    tr = build_training_frame(L[L.day < val_day], cfg, counters, logs)
    va = build_training_frame(L[(L.day == val_day) & (L.explore == 0)], cfg, counters, logs)
    model = train_ranker(tr)
    va = va[va.label.isin([0, 1])]
    auc = roc_auc_score(va.label, model.predict_proba(va[FEATS])[:, 1])
    return {"model": model, "counters": counters, "auc_offline": auc, "train_frame": tr, "val_frame": va, "cfg": cfg}


EXPLORE_WEIGHT = 5.0

In [ ]:
def simulate_online(pipe, days=N_ONLINE_DAYS, seed_=7, policy="v2", log_requests=True):
    """Producción: CineMatch sirve a los usuarios durante `days` días. Devuelve métricas por petición y un log
    (log-and-wait) con las features EXACTAS servidas y el estado de la caché. Las métricas «reales» usan el gemelo digital."""
    cfg, model, counters = pipe["cfg"], pipe["model"], pipe["counters"]
    r = np.random.default_rng(seed_)
    hist = {u: list(h) for u, h in hist_after_logs.items()}
    recent = set(logs[logs.day >= N_LOG_DAYS - 3].user_id)
    cache = {u: {"emb": user_embedding(hist[u], "v1"), "version": "v1", "hist": list(hist[u]), "ts": N_LOG_DAYS - 0.5} for u in recent}
    res, served_log = [], []
    for day in range(N_LOG_DAYS, N_LOG_DAYS + days):
        for u in r.choice(np.arange(1, N_U), size=int(REQ_FRAC * N_U), replace=False):
            t = day + r.random()
            if policy == "v1":
                top, _ = v1_policy(u, day, hist[u], r=None); ent = None
            else:
                ent = cache.get(u)
                if ent is None or (cfg["cache_key"] != "user_id" and ent["version"] != SERVING_VERSION):
                    ent = {"emb": user_embedding(hist[u], SERVING_VERSION), "version": SERVING_VERSION, "hist": list(hist[u]), "ts": t}
                    cache[u] = ent
                cands = retrieve(ent["emb"], day, ent["hist"], k=100)
                f = features(cands, day, ent["hist"], ent["emb"], cfg, counters, serving=True)
                order = np.argsort(-model.predict_proba(f[FEATS])[:, 1])
                top = cands[order[:10]]
                if log_requests:
                    served_log.append(f.assign(user_id=u, day=day, t=t, item_id=cands, served=np.isin(np.arange(len(cands)), order[:10]),
                                               cache_version=ent["version"], cache_age=t - ent["ts"], hist_len=len(ent["hist"]),
                                               true_hist_len=len(hist[u])))
            already = np.isin(top, hist[u])
            p_click = POS_BIAS * sigmoid(true_click_logit(u, top, day)) * np.where(already, 0.1, 1.0)   # re-recomendar lo visto casi no rinde
            exp_conv = float((p_click * true_conv_prob(u, top)).sum())
            clk = r.random(10) < p_click
            hist[u] += [int(i) for i in top[clk]]
            if policy != "v1" and cfg["cache_invalidate_on_event"] and clk.any():
                cache.pop(u, None)
            res.append({"day": day, "user_id": u, "conv_esperadas": exp_conv, "ya_visto": already.mean(),
                        "estrenos": np.mean(day - release[top] < 7), "cache_v1": ent is not None and ent["version"] == "v1"})
    return OnlineRun(pd.DataFrame(res), pd.concat(served_log, ignore_index=True) if served_log else None, cache, hist)


class OnlineRun:
    """Resultado de producción: métricas por petición (se accede como a un DataFrame) + log servido, caché e historiales."""
    def __init__(self, df, served_log, cache, hist):
        self.df, self.served_log, self.cache, self.hist = df, served_log, cache, hist

    def __getattr__(self, name):
        return getattr(self.df, name)


def evaluate(cfg, label=""):
    pipe = run_pipeline(cfg)
    on = simulate_online(pipe)
    return pipe, on, {"config": label, "AUC offline": pipe["auc_offline"], "conv/petición": on.conv_esperadas.mean(),
                      "% ya visto": on.ya_visto.mean(), "% estrenos": on.estrenos.mean()}

## 3. Reproduce el incidente

Comparamos v1 (lo que había) con v2 tal como se lanzó. La «AUC offline» es la que vio la revisión de lanzamiento; la
«conv/petición» es la métrica online (conversiones esperadas por petición, con el gemelo digital).

In [ ]:
t0 = time.time()
pipe_prod, online_prod, row_prod = evaluate(PROD_CONFIG, "v2 producción")
online_v1 = simulate_online(pipe_prod, policy="v1")
def v1_scores(frame):
    """Score heurístico de v1 sobre las mismas filas de validación (para comparar AUC offline)."""
    out = pd.Series(0.0, index=frame.index)
    clicks = logs[logs.click == 1]
    for (u, day), g in frame.groupby(["user_id", "day"]):
        h = history[u] + clicks.item_id[(clicks.user_id == u) & (clicks.day < day)].tolist()
        out[g.index] = pop_z[g.item_id.values] + 3.0 * (EMB["v1"][g.item_id.values] @ user_embedding(h, "v1"))
    return out
v1_auc = roc_auc_score(pipe_prod["val_frame"].label, v1_scores(pipe_prod["val_frame"]))
incident = pd.DataFrame([{"config": "v1 (anterior)", "AUC offline": v1_auc, "conv/petición": online_v1.conv_esperadas.mean(),
                          "% ya visto": online_v1.ya_visto.mean(), "% estrenos": online_v1.estrenos.mean()}, row_prod]).set_index("config")
display(incident.round(4)); print(f"({time.time() - t0:.0f} s)")
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
incident["AUC offline"].plot.bar(ax=ax[0], color=["#94a3b8", "#2563eb"], rot=0, ylim=(0.5, 1), title="Lo que vio la revisión de lanzamiento")
for name, on in [("v1", online_v1), ("v2 producción", online_prod)]:
    ax[1].plot(on.groupby("day").conv_esperadas.mean(), "o-", label=name)
ax[1].set(title="Lo que ve el pager: conv/petición por día", xlabel="día"); ax[1].legend(); plt.tight_layout(); plt.show()

*(La AUC de v1 es la de su score heurístico sobre los mismos ejemplos de validación.)*

**Offline ↑, online ↓.** A partir de aquí, trabajo de guardia.

## 4. Tu caja de herramientas de diagnóstico (TODOs)

Implementa estas seis herramientas. Son genéricas: te servirán en cualquier sistema real. Cada una debe devolver
**números** que permitan decir «verde / rojo».

In [ ]:
def cache_audit(online_df) -> dict:
    """Herramienta 1 · ¿Sirves el modelo que crees?
    Devuelve: % de peticiones servidas con vector de usuario de una versión distinta de SERVING_VERSION,
    % de recomendaciones ya vistas y la similitud coseno media entre el vector cacheado y uno recalculado
    ahora con la versión correcta e historial completo (online_df.cache, online_df.hist)."""
    # TODO
    raise NotImplementedError


def skew_report(online_df, pipe, n_sample=3000) -> pd.DataFrame:
    """Herramienta 2 · Log-and-wait: para una muestra de filas servidas (online_df.served_log, served=True),
    recalcula las features con el PIPELINE OFFLINE (features(..., serving=False) con la config de pipe, el historial
    real del usuario en ese instante y su embedding v2 recalculado) y compara fila a fila.
    Devuelve por feature: % filas distintas (np.isclose, rtol=1e-3) y la mediana del cociente servido/offline.
    Pista: el historial real en el instante de la petición es online_df.hist[u][:true_hist_len]."""
    # TODO
    raise NotImplementedError


def time_travel_audit(pipe, n_sample=5000) -> pd.DataFrame:
    """Herramienta 3 · Para filas de pipe["train_frame"], recalcula las features de contadores ESTRICTAMENTE
    point-in-time (solo datos anteriores al día de la impresión) y compara con lo que usó el entrenamiento.
    Devuelve por feature: % filas distintas y AUC univariante de la feature contra la etiqueta en ambas versiones."""
    # TODO
    raise NotImplementedError


def hash_collision_audit(counters, logs) -> dict:
    """Herramienta 4 · Nº de películas activas por fila de la tabla, contaminación ponderada por impresiones
    (fracción de las impresiones de la fila que son de OTRAS películas) y correlación entre el CTR de la fila y el
    CTR exacto de la película."""
    # TODO
    raise NotImplementedError


def label_maturity_curve(logs, T=T_TRAIN, bins=np.arange(0, 15)) -> pd.DataFrame:
    """Herramienta 5 · Para los CLICS, tasa de conversión observada en T según la edad del clic (T − t), por
    tramos de 1 día. Si la tasa cae para los clics recientes, hay label delay en el set de entrenamiento."""
    # TODO
    raise NotImplementedError


def adversarial_validation(train_frame, served_log, feats=None) -> dict:
    """Herramienta 6 · Entrena un clasificador que distinga filas de entrenamiento de candidatos de serving
    (served_log, TODOS los candidatos, no solo los servidos). Devuelve la AUC y la AUC univariante por feature.
    AUC ≈ 0,5 → misma distribución. Excluye features con skew ya conocido o con distinta semántica (position)."""
    # TODO
    raise NotImplementedError

## 5. Bug por bug (en el orden del árbol de triaje)

Para **cuantificar**, usa este *helper*: arregla **un solo** parámetro sobre la config de producción y mide Δ online.

In [ ]:
def quantify(change: dict, base=PROD_CONFIG, label=None):
    cfg = {**base, **change}
    _, on, row = evaluate(cfg, label or str(change))
    base_val = incident.loc["v2 producción", "conv/petición"]
    row["Δ conv/petición vs prod"] = row["conv/petición"] / base_val - 1
    return on, row

### 🚨 Bug 1 · ¿Sirves el modelo que crees?
El pager dice que la caída empezó **justo** con el despliegue. Primer paso del triaje: comprueba versión y caché.

💡 *Pista:* Usa `cache_audit(online_prod)`. Arreglo: en la config hay dos parámetros de caché.

In [ ]:
# TODO bug 1: (1) detecta con la herramienta y muestra el número que lo delata,
#               (2) cuantifica con quantify({...}), (3) deja aquí el cambio de config que lo arregla.
FIX_1 = {}   # p. ej. {"parametro": valor}

### 🚨 Bug 2 · ¿Las features que sirves son las que entrenaste?
Con la caché bajo control, compara las features servidas con las recalculadas offline (*log-and-wait*).

💡 *Pista:* Usa `skew_report(online_prod, pipe_prod)`. Varias features saldrán en rojo (algunas son síntomas de los bugs 1 y 3); el bug 2 es la que tiene un **factor constante** entre ambos lados.

In [ ]:
# TODO bug 2: (1) detecta con la herramienta y muestra el número que lo delata,
#               (2) cuantifica con quantify({...}), (3) deja aquí el cambio de config que lo arregla.
FIX_2 = {}   # p. ej. {"parametro": valor}

### 🚨 Bug 3 · ¿La evaluación offline es honesta?
La AUC offline de v2 es sospechosamente alta. Audita las features de contadores con *time-travel*.

💡 *Pista:* Usa `time_travel_audit(pipe_prod)`. ¿Qué ventana usa el pipeline de train vs el servicio?

In [ ]:
# TODO bug 3: (1) detecta con la herramienta y muestra el número que lo delata,
#               (2) cuantifica con quantify({...}), (3) deja aquí el cambio de config que lo arregla.
FIX_3 = {}   # p. ej. {"parametro": valor}

### 🚨 Bug 4 · Contadores que no distinguen películas
Los contadores de CTR/CVR por película parecen poco informativos. Audita la tabla.

💡 *Pista:* Usa `hash_collision_audit(pipe_prod['counters'], logs)`.

In [ ]:
# TODO bug 4: (1) detecta con la herramienta y muestra el número que lo delata,
#               (2) cuantifica con quantify({...}), (3) deja aquí el cambio de config que lo arregla.
FIX_4 = {}   # p. ej. {"parametro": valor}

### 🚨 Bug 5 · Etiquetas que aún no han llegado
La conversión tarda días. ¿Qué pasa con los clics de los últimos días del set de entrenamiento?

💡 *Pista:* Usa `label_maturity_curve(logs)`. Arreglo: implementa `soft_labels(L, T)` (posterior de convertir dado que aún no ha convertido, modelo de Chapelle con retraso exponencial) y usa `label_policy='soft'`.

In [ ]:
# TODO bug 5: (1) detecta con la herramienta y muestra el número que lo delata,
#               (2) cuantifica con quantify({...}), (3) deja aquí el cambio de config que lo arregla.
FIX_5 = {}   # p. ej. {"parametro": valor}

### 🚨 Bug 6 · El ranker nunca vio estos candidatos
v2 tiene un retriever nuevo que no aplica el prefiltro de popularidad de v1. ¿Con qué datos aprendió el ranker?

💡 *Pista:* Usa `adversarial_validation(pipe['train_frame'], served_log)` (con los bugs 1–2 ya arreglados para que no contaminen la comparación). Arreglo: usa también las impresiones de exploración, ponderadas (`train_on='all_ipw'`).

In [ ]:
# TODO bug 6: (1) detecta con la herramienta y muestra el número que lo delata,
#               (2) cuantifica con quantify({...}), (3) deja aquí el cambio de config que lo arregla.
FIX_6 = {}   # p. ej. {"parametro": valor}

In [ ]:
def soft_labels(L, T):
    """TODO (bug 5): para clics con label == 0, sustituye la etiqueta por P(convertirá | aún no convirtió tras e = T − t)
    = p·S(e) / (1 − p + p·S(e)), con S(e) = exp(−e/μ). Estima p y μ con clics maduros (t ≤ T − 7)."""
    raise NotImplementedError

## 6. *Scorecard*

In [ ]:
FIXED_CONFIG = {**PROD_CONFIG, **FIX_1, **FIX_2, **FIX_3, **FIX_4, **FIX_5, **FIX_6}


def scorecard(fixed_cfg):
    pipe, on, row = evaluate(fixed_cfg, "v2 arreglado")
    v1v, prodv = incident.loc["v1 (anterior)", "conv/petición"], incident.loc["v2 producción", "conv/petición"]
    honest = {**fixed_cfg, "counter_window": "pit"}
    auc_honest = run_pipeline(honest)["auc_offline"]
    checks = {
        "conv/petición ≥ 1,10 × producción": row["conv/petición"] >= 1.10 * prodv,
        "conv/petición ≥ 0,98 × v1": row["conv/petición"] >= 0.98 * v1v,
        "% ya visto < 1 %": row["% ya visto"] < 0.01,
        "AUC offline honesta (|Δ| < 0,01)": abs(row["AUC offline"] - auc_honest) < 0.01,
    }
    display(pd.DataFrame([incident.loc["v1 (anterior)"].to_dict() | {"config": "v1"}, incident.loc["v2 producción"].to_dict() | {"config": "v2 prod"}, row]).set_index("config").round(4))
    for k_, v in checks.items():
        print(("✅ " if v else "❌ ") + k_)
    return pipe, on, row

try:
    _ = scorecard(FIXED_CONFIG)
except NotImplementedError:
    print("Implementa las herramientas y los arreglos para ver el scorecard.")

## 7. *Postmortem* (plantilla)

```text
Título: Caída de conversiones tras el lanzamiento de CineMatch v2
Severidad: P1 · Duración: … · Impacto: −…% conv/petición (… conversiones perdidas)
Línea temporal: lanzamiento → alerta → mitigación → arreglos
Causas raíz (una por bug): qué, por qué pasó, por qué no lo vio la revisión de lanzamiento
Qué funcionó / qué no
Acciones (con owner y fecha):  p. ej. golden queries en el CI de despliegue; skew report diario con log-and-wait;
  contadores PIT por construcción (ventana semiabierta en la librería); curva de maduración en el dashboard del
  pipeline; adversarial validation train-vs-serving como gate; tabla de contadores sin hash…
```

---
# ⛔ SPOILER — intenta resolverlo primero

Solución de referencia completa: las seis herramientas, el diagnóstico de cada bug con su número, la cuantificación
(arreglar uno sobre producción **y** romper uno sobre el sistema arreglado — los bugs interactúan) y el *scorecard*.

In [ ]:
def cache_audit(online_df):
    cache, hist = online_df.cache, online_df.hist
    cos = [float(e["emb"] @ user_embedding(hist[u], SERVING_VERSION)) for u, e in cache.items()]
    stale_hist = np.mean([len(e["hist"]) < len(hist[u]) for u, e in cache.items()])
    return {"% peticiones con vector de otra versión": online_df.cache_v1.mean(),
            "% recomendaciones ya vistas": online_df.ya_visto.mean(),
            "coseno medio caché vs recalculado": float(np.mean(cos)),
            "% entradas con historial desactualizado": float(stale_hist)}


def skew_report(online_df, pipe, n_sample=3000):
    sl_all, final_hist = online_df.served_log, online_df.hist
    sl = sl_all[sl_all.served].sample(n_sample, random_state=0)
    rows = []
    for (u, day, t), g in sl.groupby(["user_id", "day", "t"]):
        h = final_hist[u][: g.true_hist_len.iloc[0]]                    # historial real en el instante de la petición
        f = features(g.item_id.values, day, h, user_embedding(h, SERVING_VERSION), pipe["cfg"], pipe["counters"], serving=False)
        rows.append(f.set_index(g.index))
    off = pd.concat(rows).loc[sl.index]
    rep = []
    for c_ in [f_ for f_ in FEATS if f_ != "position"]:
        a, b = sl[c_].values, off[c_].values
        mism = ~np.isclose(a, b, rtol=1e-3, atol=1e-6)
        ratio = np.median(a[mism] / np.where(b[mism] == 0, np.nan, b[mism])) if mism.any() else np.nan
        rep.append({"feature": c_, "% filas distintas": mism.mean(), "mediana servido/offline": ratio})
    out = pd.DataFrame(rep).set_index("feature")
    out["veredicto"] = np.where(out["% filas distintas"] > 0.02, "🔴", "🟢")
    return out

In [ ]:
def time_travel_audit(pipe, n_sample=5000):
    tr = pipe["train_frame"]; tr = tr[tr.label.isin([0, 1])]
    tr = tr.sample(min(n_sample, len(tr)), random_state=0)
    ctr, cvr, imps = [], [], []
    for day, g in tr.groupby("day"):
        a, b, c_ = pipe["counters"].asof(g.item_id.values, int(day), inclusive=False)
        ctr.append(pd.Series(a, g.index)); cvr.append(pd.Series(b, g.index)); imps.append(pd.Series(c_, g.index))
    pit = pd.DataFrame({"item_ctr_1d": pd.concat(ctr), "item_cvr_7d": pd.concat(cvr), "item_imps_7d": pd.concat(imps)}).loc[tr.index]
    rep = []
    for c_ in pit.columns:
        mism = ~np.isclose(tr[c_].values, pit[c_].values, rtol=1e-6)
        rep.append({"feature": c_, "% filas distintas": mism.mean(), "AUC usada en train": roc_auc_score(tr.label, tr[c_]),
                    "AUC point-in-time": roc_auc_score(tr.label, pit[c_])})
    return pd.DataFrame(rep).set_index("feature")


def hash_collision_audit(counters, logs):
    imps = logs.item_id.value_counts()
    b = counters.bucket(imps.index.values)
    row_imps = pd.Series(imps.values).groupby(b).transform("sum").values
    contamination = 1 - imps.values / row_imps
    ctr_exact = logs.groupby("item_id").click.mean().loc[imps.index].values
    row_ctr = pd.Series(logs.click.values).groupby(counters.bucket(logs.item_id.values)).mean()
    return {"películas activas": len(imps), "filas usadas": int(np.unique(b).size),
            "películas por fila (media)": len(imps) / np.unique(b).size,
            "contaminación ponderada por impresiones": float((imps.values * contamination).sum() / imps.values.sum()),
            "corr(CTR fila, CTR exacto)": float(np.corrcoef(row_ctr.loc[b].values, ctr_exact)[0, 1])}


def label_maturity_curve(logs, T=T_TRAIN, bins=np.arange(0, 15)):
    c_ = logs[logs.click == 1].copy()
    c_["edad_clic"] = T - c_.t
    c_["observada"] = (c_.conv == 1) & (c_.conv_t <= T)
    c_["tramo"] = pd.cut(c_.edad_clic, np.r_[bins, np.inf], right=False, labels=bins)
    return c_.groupby("tramo", observed=True).agg(clics=("observada", "size"), cvr_observada=("observada", "mean"),
                                                 cvr_final_oraculo=("conv", "mean"))


def adversarial_validation(train_frame, served_log, feats=None):
    feats = feats or ["retr_score", "item_ctr_1d", "item_cvr_7d", "item_imps_7d", "item_pop_hist", "user_n_hist"]
    a = train_frame[feats].sample(min(20_000, len(train_frame)), random_state=0)
    b = served_log[feats].sample(min(20_000, len(served_log)), random_state=0)
    Xa = pd.concat([a, b]); ya = np.r_[np.zeros(len(a)), np.ones(len(b))]
    from sklearn.model_selection import cross_val_predict
    p = cross_val_predict(HistGradientBoostingClassifier(max_iter=100, random_state=seed), Xa, ya, cv=3, method="predict_proba")[:, 1]
    uni = {f_: max(roc_auc_score(ya, Xa[f_]), 1 - roc_auc_score(ya, Xa[f_])) for f_ in feats}
    return {"AUC train-vs-serving": roc_auc_score(ya, p), "AUC univariante": pd.Series(uni).sort_values(ascending=False).round(3)}


def soft_labels(L, T):
    mature = L[(L.click == 1) & (L.t <= T - 7)]
    p = mature.label.mean()
    mu = (mature.conv_t - mature.t)[mature.label == 1].mean()
    pending = (L.click == 1) & (L.label == 0)
    surv = np.exp(-(T - L.t) / mu)
    L.loc[pending, "label"] = (p * surv / (1 - p + p * surv))[pending]
    return L

### Diagnóstico bug por bug

In [ ]:
t0 = time.time()
print("🚨 Bug 1 · caché:", {k_: round(v, 3) for k_, v in cache_audit(online_prod).items()})
FIX_1 = {"cache_key": "user_id+version", "cache_invalidate_on_event": True}
print("🚨 Bug 2 · skew (log-and-wait):"); display(skew_report(online_prod, pipe_prod).round(3))
FIX_2 = {"age_unit_serving": "days"}
print("🚨 Bug 3 · time-travel:"); display(time_travel_audit(pipe_prod).round(3))
FIX_3 = {"counter_window": "pit"}
print("🚨 Bug 4 · hash:", {k_: round(v, 3) for k_, v in hash_collision_audit(pipe_prod["counters"], logs).items()})
FIX_4 = {"hash_buckets": None}
print(f"({time.time() - t0:.0f} s)")

In [ ]:
mat = label_maturity_curve(logs)
print("🚨 Bug 5 · maduración de la etiqueta:"); display(mat.round(3).T)
FIX_5 = {"label_policy": "soft"}
ax = mat[["cvr_observada", "cvr_final_oraculo"]].plot(marker="o", figsize=(8, 3.4), title="Curva de maduración: los clics recientes parecen negativos")
ax.set(xlabel="edad del clic en el momento de entrenar (días)", ylabel="conversiones / clic"); plt.show()

pipe_12 = run_pipeline({**PROD_CONFIG, **FIX_1, **FIX_2})
on_12 = simulate_online(pipe_12)
av_bad = adversarial_validation(pipe_12["train_frame"], on_12.served_log)
pipe_6 = run_pipeline({**PROD_CONFIG, **FIX_1, **FIX_2, "train_on": "all_ipw"})
av_good = adversarial_validation(pipe_6["train_frame"], on_12.served_log)
print(f"🚨 Bug 6 · adversarial validation: AUC train-vs-serving {av_bad['AUC train-vs-serving']:.3f} (solo impresiones) "
      f"→ {av_good['AUC train-vs-serving']:.3f} (con exploración ponderada)")
display(pd.DataFrame({"solo impresiones": av_bad["AUC univariante"], "con exploración": av_good["AUC univariante"]}))
FIX_6 = {"train_on": "all_ipw"}

### Cuantificación: arreglar uno sobre producción y romper uno sobre el sistema arreglado

Los bugs **interactúan** (p. ej. con la caché rota, el resto apenas importa porque el retriever ya devuelve basura).
Por eso un *postmortem* serio da las dos vistas.

In [ ]:
t0 = time.time()
FIXES = {"1 caché": FIX_1, "2 skew edad": FIX_2, "3 fuga contadores": FIX_3, "4 hash": FIX_4, "5 label delay": FIX_5, "6 candidate mismatch": FIX_6}
FIXED_CONFIG = {**PROD_CONFIG, **FIX_1, **FIX_2, **FIX_3, **FIX_4, **FIX_5, **FIX_6}
pipe_fix, online_fix, row_fix = evaluate(FIXED_CONFIG, "todo arreglado")
rows = []
for name, fx in FIXES.items():
    _, r1 = quantify(fx, label=name)
    broken = {**FIXED_CONFIG, **{k_: PROD_CONFIG[k_] for k_ in fx}}
    _, _, r2 = evaluate(broken, name)
    rows.append({"bug": name, "Δ al arreglar solo este (sobre prod)": r1["Δ conv/petición vs prod"],
                 "Δ al romper solo este (sobre arreglado)": r2["conv/petición"] / row_fix["conv/petición"] - 1,
                 "AUC offline con el bug (resto arreglado)": r2["AUC offline"]})
impact = pd.DataFrame(rows).set_index("bug"); display(impact.round(4))
print(f"({time.time() - t0:.0f} s)")
ax = impact.iloc[:, :2].mul(100).plot.barh(figsize=(9, 4), title="Impacto de cada bug en conv/petición (%)")
ax.axvline(0, color="k", lw=0.8); ax.set_xlabel("%"); plt.show()

In [ ]:
pipe_fix, online_fix, row_fix = scorecard(FIXED_CONFIG)
fig, ax = plt.subplots(figsize=(8, 3.6))
for name, on in [("v1", online_v1), ("v2 producción", online_prod), ("v2 arreglado", online_fix)]:
    ax.plot(on.groupby("day").conv_esperadas.mean(), "o-", label=name)
ax.set(title="Después del incidente", xlabel="día", ylabel="conv/petición"); ax.legend(); plt.show()

### Lectura de la solución (con FAST_DEV_RUN; tus números variarán algo)

1. **Caché**: casi todos los usuarios recientes se sirven con su vector **v1** contra el índice **v2**; como v2 viene
   de un *warm-start* el espacio está parcialmente alineado y el sistema «funciona» peor sin dar error. Además, sin
   invalidación por evento, el filtro de «ya visto» usa un historial congelado. Arreglo: versión en la clave +
   invalidación por evento (y en un sistema real: *golden queries* en el despliegue).
2. **Skew**: `item_age_days` servido = 24 × offline (horas vs días). Los estrenos parecen viejos y desaparecen de la
   home (mira `% estrenos`).
3. **Fuga**: el pipeline de train agrega los contadores **incluyendo el día de la impresión** (ventana inclusiva):
   el CTR y la CVR «del último día» contienen el propio clic/conversión. La AUC univariante de esas features cae al
   recalcularlas PIT, y la AUC offline del modelo baja: la revisión de lanzamiento estaba viendo una fuga.
4. **Hash**: 97 filas para ~1 600 películas → ~15 películas por fila; la CTR de la fila apenas correlaciona con la de la
   película. Su impacto aquí es **pequeño** (≈ −0,5–1 %): con pocos clics por película los contadores exactos ya son
   ruidosos y otras features compensan. En catálogos de millones, donde los contadores son features principales, el
   mismo bug cuesta mucho más. Lección de guardia: **prioriza por impacto medido**, no por lo feo que es el bug — y aun
   así arréglalo, porque la tabla exacta no cuesta nada con este catálogo.
5. **Label delay**: la tasa de conversión observada de los clics de los últimos 3–4 días es una fracción de la real
   (curva de maduración). El ranker aprende que lo reciente —los estrenos— no convierte. Las etiquetas suaves de
   Chapelle lo corrigen sin tirar los datos frescos.
6. **Candidate mismatch**: el ranker solo vio películas populares (el prefiltro de v1); el retriever v2 trae cola larga
   con mucho «relleno» de baja calidad que el ranker no sabe penalizar. La *adversarial validation* lo delata
   (`item_pop_hist` separa train de serving) y usar el tráfico de exploración ponderado lo corrige.

## 🚀 Retos extra (nivel experto)
- Sustituye `EXPLORE_WEIGHT` por pesos IPS exactos a partir de la propensión de cada hueco de exploración y compara.
- Implementa *golden queries* para el despliegue: 200 usuarios con vecinos conocidos en v2; el despliegue falla si el
  recall@20 cae > 5 %. Comprueba que habría parado el bug 1 antes de llegar a usuarios.
- Haz que el *skew report* corra «cada noche» sobre el log de un día y dispare una alerta (módulo 17).
- Añade un bug 7 tuyo (p. ej. orden del historial invertido en serving) y comprueba qué herramienta lo detecta.
- Estima con OPE (módulo 14) el valor de v2 arreglado usando solo el tráfico de exploración, antes de desplegar.

## 🤔 Reflexión
1. ¿Qué tres *gates* automáticos en el CI/CD habrían evitado este incidente? ¿Cuánto cuestan?
2. ¿Por qué la AUC offline no detectó ninguno de los bugs de *serving* (1, 2) y sí «premió» el 3?
3. Si solo pudieras tener **una** de las seis herramientas en producción, ¿cuál y por qué?
4. ¿Cómo cambiaría tu respuesta al bug 5 si el retraso medio fuese de 14 días? ¿Y de 1 hora?
5. ¿Qué parte del tráfico de exploración «pagas» y cómo lo justificas ante producto?